# Energy Consumption Prediction

**Tabular Regression Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `energy_kwh`

## 2 · Project Overview

We predict **building energy consumption** (kWh) based on weather conditions (temperature, humidity, wind), building characteristics (size, type), occupancy, and temporal features (hour, day).

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Encode categorical features for tree-based and linear models.
3. Build a baseline Linear Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with RMSE, MAE, R², and residual analysis.

## 4 · Problem Statement

Given weather conditions, building size and type, occupancy, and time of day, predict hourly energy consumption in kWh.

## 5 · Why This Project Matters

- **Energy management** reduces costs and carbon emissions.
- Demand forecasting supports grid load balancing.
- Building efficiency optimization starts with consumption prediction.
- Demonstrates regression with non-linear temperature effects.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 8,000 |
| **Features** | temperature_f, humidity_pct, wind_speed_mph, building_sqft, num_occupants, hour_of_day, day_of_week, is_weekend, building_type |
| **Target** | energy_kwh (continuous) |
| **Range** | ~10 – 10,000 kWh |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "energy_kwh"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

## 11 · Dataset Download or Loading

Synthetic dataset: 8,000 building energy consumption records with weather and occupancy features.

In [ ]:
np.random.seed(SEED)
n = 8000
temperature_f = np.round(np.random.normal(60, 20, n).clip(-10, 115), 1)
humidity_pct = np.round(np.random.uniform(15, 95, n), 1)
wind_speed_mph = np.round(np.random.exponential(8, n).clip(0, 50), 1)
building_sqft = np.random.choice([5000, 10000, 20000, 50000, 100000], n, p=[0.2, 0.3, 0.25, 0.15, 0.1])
num_occupants = np.random.poisson(50, n).clip(5, 500)
hour_of_day = np.random.randint(0, 24, n)
day_of_week = np.random.randint(0, 7, n)
is_weekend = (day_of_week >= 5).astype(int)
building_type = np.random.choice(["Office", "Retail", "Hospital", "School", "Warehouse"], n,
                                  p=[0.3, 0.2, 0.15, 0.2, 0.15])
btype_mult = {"Office": 1.0, "Retail": 0.9, "Hospital": 1.5, "School": 0.8, "Warehouse": 0.6}
bt_val = np.array([btype_mult[b] for b in building_type])

# Energy model: HVAC load (temp deviation) + occupancy + base load
temp_deviation = np.abs(temperature_f - 68)
hvac_load = 0.5 * temp_deviation * (building_sqft / 1000)
occupancy_load = 0.3 * num_occupants
hour_factor = 0.5 + 0.5 * np.sin(np.pi * (hour_of_day - 6) / 12).clip(0, 1)
base_load = building_sqft * 0.02

energy_kwh = (base_load + hvac_load + occupancy_load) * bt_val * (1 - 0.3 * is_weekend) * hour_factor
energy_kwh = np.round(energy_kwh + np.random.normal(0, 50, n), 1).clip(10, 10000)

df = pd.DataFrame({"temperature_f": temperature_f, "humidity_pct": humidity_pct,
                    "wind_speed_mph": wind_speed_mph, "building_sqft": building_sqft,
                    "num_occupants": num_occupants, "hour_of_day": hour_of_day,
                    "day_of_week": day_of_week, "is_weekend": is_weekend,
                    "building_type": building_type, "energy_kwh": energy_kwh})
print(f"Dataset shape: {df.shape}")
print(f"Target stats:\n{df['energy_kwh'].describe()}")
df.head()

## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")
print(f"\nTarget stats:\n{df[TARGET].describe()}")

## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0][0].scatter(df["temperature_f"], df[TARGET], alpha=0.2, s=10)
axes[0][0].set_xlabel("Temperature (°F)"); axes[0][0].set_ylabel("Energy (kWh)")
axes[0][0].set_title("Temperature vs Energy")

axes[0][1].scatter(df["building_sqft"], df[TARGET], alpha=0.2, s=10)
axes[0][1].set_xlabel("Building Sqft"); axes[0][1].set_ylabel("Energy (kWh)")
axes[0][1].set_title("Building Size vs Energy")

axes[0][2].scatter(df["num_occupants"], df[TARGET], alpha=0.2, s=10)
axes[0][2].set_xlabel("Occupants"); axes[0][2].set_ylabel("Energy (kWh)")
axes[0][2].set_title("Occupants vs Energy")

df.groupby("hour_of_day")[TARGET].mean().plot(ax=axes[1][0], marker="o", color="steelblue")
axes[1][0].set_title("Avg Energy by Hour"); axes[1][0].set_xlabel("Hour")

df.boxplot(column=TARGET, by="building_type", ax=axes[1][1])
axes[1][1].set_title("Energy by Building Type")
axes[1][1].tick_params(axis="x", rotation=45)

corr = df.select_dtypes(include="number").corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1][2], square=True, cbar_kws={"shrink":0.8})
axes[1][2].set_title("Correlation")

plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `energy_kwh`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[TARGET], bins=30, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}")
axes[0].set_xlabel("Energy (kWh)")

df.boxplot(column=TARGET, by="is_weekend", ax=axes[1])
axes[1].set_title("Energy: Weekday vs Weekend")

plt.tight_layout()
plt.show()
print(f"Range: {df[TARGET].min():,.1f} to {df[TARGET].max():,.1f} kWh")
print(f"Mean: {df[TARGET].mean():,.1f}, Median: {df[TARGET].median():,.1f}")
print(f"Weekend reduction: {1 - df[df['is_weekend']==1][TARGET].mean()/df[df['is_weekend']==0][TARGET].mean():.1%}")

## 15 · Train/Validation/Test Split Strategy

80/20 train/test split.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target — Train mean: {y_train.mean():,.2f}, Test mean: {y_test.mean():,.2f}")

## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `building_type` encoded with OrdinalEncoder.
- **Scaling**: Not needed for tree models.
- **Temporal**: `hour_of_day` and `day_of_week` kept as integers.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [ ]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["temp_deviation"] = np.abs(X_train["temperature_f"] - 68)
X_test["temp_deviation"] = np.abs(X_test["temperature_f"] - 68)

X_train["hour_sin"] = np.sin(2 * np.pi * X_train["hour_of_day"] / 24)
X_test["hour_sin"] = np.sin(2 * np.pi * X_test["hour_of_day"] / 24)

X_train["hour_cos"] = np.cos(2 * np.pi * X_train["hour_of_day"] / 24)
X_test["hour_cos"] = np.cos(2 * np.pi * X_test["hour_of_day"] / 24)

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

## 18 · Baseline Model

Linear Regression as our baseline regressor.

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

## 19 · LazyPredict Benchmark

Fit dozens of regressors in one call for a quick ranking.

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Regressors:")
print(lazy_models.head(15).to_string())

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [ ]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="regression", time_budget=60,
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred_flaml):,.2f}")
print(f"R2  : {r2_score(y_test, y_pred_flaml):.4f}")

## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [ ]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    try:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    try:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Linear Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)

    axes[i].scatter(y_test, yp, alpha=0.6, s=40, edgecolors="black")
    mn, mx = min(y_test.min(), yp.min()), max(y_test.max(), yp.max())
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.2f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_predicted_vs_actual.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")

## 24 · Error Analysis

Examine residuals from the best model.

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=20, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.6, s=40, edgecolors="black")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

sorted_idx = np.argsort(best_pred)
axes[2].scatter(best_pred[sorted_idx], np.abs(residuals[sorted_idx]), alpha=0.6, s=40, edgecolors="black")
axes[2].set_title(f"{best_name} — |Residual| vs Predicted")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("|Residual|")

plt.tight_layout()
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")
print(f"  Median: {np.median(residuals):,.2f}")

## 25 · Interpretation and Business Insight

**Key findings:**
- **Building size** is the strongest energy predictor (base load).
- **Temperature deviation** from 68°F drives HVAC load non-linearly.
- **Occupancy** adds linear incremental energy.
- **Hospital** consumes 1.5× more than offices (24/7 operations).
- **Weekends** reduce energy by ~30%.

**Business takeaway:** HVAC optimization (setpoint tuning) and weekend/off-hours scheduling are the biggest savings levers.

## 26 · Limitations

1. Synthetic data with simplified energy dynamics.
2. No HVAC system type or efficiency rating.
3. Missing solar gain and insulation quality.
4. No sub-metering (lighting vs. HVAC vs. plug loads).
5. No seasonal trend or year-over-year comparison.

## 27 · How to Improve This Project

1. Use real smart-meter data.
2. Add HVAC system specifications.
3. Include building envelope properties.
4. Model as time series with lagged features.
5. Add utility rate structures for cost optimization.

## 28 · Production Considerations

- Deploy for real-time energy monitoring dashboards.
- Integrate with building management systems (BMS).
- Trigger alerts on anomalous consumption.
- Update monthly with new calibration data.
- Output demand forecasts for grid operators.

## 29 · Common Mistakes

1. Treating temperature as linearly related to energy (it's V-shaped).
2. Ignoring the cyclical nature of hour/day features.
3. Not separating base load from variable load.
4. Using future weather in real-time predictions.
5. Not accounting for building-specific baselines.

## 30 · Mini Challenge / Exercises

1. Plot energy vs. temperature — confirm the V-shape.
2. Create `cooling_degree` and `heating_degree` features.
3. Build separate models by building type.
4. Remove `is_weekend` — how much does RMSE increase?
5. Try cyclical encoding vs. raw `hour_of_day`.

## 31 · Final Summary / Key Takeaways

1. **Building size** determines base energy load.
2. **Temperature deviation** drives HVAC consumption (V-shape).
3. **Occupancy** adds linear energy intensity.
4. **Building type** (Hospital > Office > Warehouse) sets efficiency profile.
5. **Weekend/off-hours** reduce consumption by ~30%.
6. **Non-linear features** (temp deviation, cyclical hour) are critical.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))